# Chapter 7: Agentic RL — Case Study

## Progressive Email Triage with RL-Learned Tool Use

This notebook demonstrates the progressive agentic architecture from Chapter 7.
An RL agent learns to triage customer support emails through six capability
levels: random baseline, LLM classification, tool use, RL learning, planning
guidance, and persistent memory.

In [ ]:
# Setup — run once
# Local:  pip install -e '.[agentic]'
# Colab:  !pip install -q "git+https://github.com/rl-textbook/reinforcement-learning.git#egg=rlbook[agentic]"

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import PPO

from rlbook.algorithms.agentic import (
    SupportAgent, CustomerMemorySystem,
    Level1RandomAgent, Level2ClassificationAgent,
    EmailTriageEnvProgressive, evaluate_rl_model,
    EMAIL_SCENARIOS,
)

## 1. LLM Agent & Scenario Data

In [ ]:
agent = SupportAgent()
print(f'Loaded {len(EMAIL_SCENARIOS)} scenarios')
repeat_count = sum(1 for s in EMAIL_SCENARIOS if s.get("is_repeat", False))
print(f'{repeat_count} repeat customer scenarios')

## 2. Pre-generate Investigation Plans

Investigation plans are generated once by the LLM and cached for use
during RL training at Levels 5-6.

In [ ]:
print('Pre-generating investigation plans...')
INVESTIGATION_PLANS = {}
for scenario in EMAIL_SCENARIOS:
    plan = agent.generate_plan(scenario['email'])
    INVESTIGATION_PLANS[scenario['id']] = plan
print(f'Generated {len(INVESTIGATION_PLANS)} plans')

## 3. Global Memory System

In [ ]:
GLOBAL_MEMORY = CustomerMemorySystem()

## 4. Training Progressive Levels (3-6)

Levels 1-2 are non-learning baselines (random and LLM classification).
Levels 3-6 use PPO to learn which investigation tools to invoke and when.

In [ ]:
models = {}

for level in [3, 4, 5, 6]:
    print(f'\n[Level {level}] Training...')
    memory = CustomerMemorySystem() if level == 6 else GLOBAL_MEMORY
    env = EmailTriageEnvProgressive(
        level=level, memory_system=memory,
        agent=agent, scenarios=EMAIL_SCENARIOS,
        investigation_plans=INVESTIGATION_PLANS if level >= 5 else {},
    )
    models[level] = PPO('MlpPolicy', env, learning_rate=3e-4, verbose=0)
    n_steps = {3: 5000, 4: 8000, 5: 10000, 6: 15000}[level]
    models[level].learn(total_timesteps=n_steps)
    print(f'  Level {level} trained for {n_steps} steps')

## 5. Evaluation Across All Levels

In [ ]:
results = {}

# Level 1: Random baseline
env_l1 = EmailTriageEnvProgressive(level=1, agent=agent, scenarios=EMAIL_SCENARIOS)
agent_l1 = Level1RandomAgent(env_l1)
results[1] = agent_l1.evaluate(100)
results[1]['memory_usage'] = 0
results[1]['memory_on_repeats'] = 0

# Level 2: LLM classification
env_l2 = EmailTriageEnvProgressive(level=2, agent=agent, scenarios=EMAIL_SCENARIOS)
agent_l2 = Level2ClassificationAgent(env_l2, agent)
results[2] = agent_l2.evaluate(100)
results[2]['memory_usage'] = 0
results[2]['memory_on_repeats'] = 0

# Levels 3-6: RL agents
for level in [3, 4, 5, 6]:
    memory = CustomerMemorySystem() if level == 6 else GLOBAL_MEMORY
    env = EmailTriageEnvProgressive(
        level=level, memory_system=memory,
        agent=agent, scenarios=EMAIL_SCENARIOS,
        investigation_plans=INVESTIGATION_PLANS if level >= 5 else {},
    )
    results[level] = evaluate_rl_model(models[level], env, n_episodes=100)

In [ ]:
# Print results table
capabilities = [
    'Random Baseline', '+ LLM Classification', '+ Tool Use',
    '+ RL Learning', '+ Planning Guidance', '+ Persistent Memory',
]

print('PROGRESSIVE COMPLEXITY RESULTS')
print('=' * 60)
for level in range(1, 7):
    print(f'\nLevel {level}: {capabilities[level-1]}')
    print(f'   Success Rate:  {results[level]["success_rate"]:.1f}%')
    print(f'   Mean Reward:   {results[level]["mean_reward"]:.3f}')
    print(f'   Tools Used:    {results[level]["mean_tools"]:.1f}')

## 6. Visualization

In [ ]:
levels = list(range(1, 7))
success_rates = [results[l]['success_rate'] for l in levels]
mean_tools = [results[l]['mean_tools'] for l in levels]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Success rate progression
ax1.plot(levels, success_rates, marker='o', linewidth=2, color='#2196F3', markersize=10)
ax1.fill_between(levels, success_rates, alpha=0.1, color='#2196F3')
ax1.set_xlabel('Capability Level', fontsize=12)
ax1.set_ylabel('Success Rate (%)', fontsize=12)
ax1.set_title('Progressive Success Rate', fontsize=14)
ax1.set_xticks(levels)
ax1.set_xticklabels([f'L{l}' for l in levels])
ax1.grid(True, alpha=0.3)

# Tool usage
ax2.bar(levels, mean_tools, color=['#FF9800' if l <= 2 else '#4CAF50' for l in levels])
ax2.set_xlabel('Capability Level', fontsize=12)
ax2.set_ylabel('Mean Tools Used', fontsize=12)
ax2.set_title('Tool Usage per Level', fontsize=14)
ax2.set_xticks(levels)
ax2.set_xticklabels([f'L{l}' for l in levels])
ax2.grid(axis='y', alpha=0.3)

plt.suptitle('Agentic RL: Progressive Capability Results', fontsize=14)
plt.tight_layout(rect=[0,0,1,0.93])
plt.show()

### Reward by Level

In [ ]:
mean_rewards = [results[l]['mean_reward'] for l in levels]

plt.figure(figsize=(8, 5))
bars = plt.bar(levels, mean_rewards,
               color=['#f44336', '#FF9800', '#FFC107', '#8BC34A', '#4CAF50', '#2196F3'])
plt.xlabel('Capability Level', fontsize=12)
plt.ylabel('Mean Reward', fontsize=12)
plt.title('Mean Reward by Capability Level', fontsize=14)
plt.xticks(levels, [f'L{l}\n{cap}' for l, cap in zip(levels, capabilities)],
           fontsize=8, rotation=15)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Key Takeaways

The progressive architecture demonstrates how each capability layer
contributes to overall performance. RL learns effective tool usage
through reward signals, while LLM planning provides semantic guidance
that accelerates learning. Persistent memory enables the agent to
handle repeat customer scenarios more effectively, showing the
compounding value of combining RL with classical AI components.